## **Imports**

In [ ]:
!jupyter nbconvert --to script communityGraphFunctions.ipynb quantumUtility.ipynb

In [ ]:
from communityGraphFunctions import *
from quantumUtility import *

In [ ]:
# Define global parameters for the problem
IBM_KEY = '' # IBM Quantum API key for authenticating with Qiskit Runtime services
# Initial Parameters for optimization
INITIAL_GAMMA = np.pi # Initial value for the gamma parameter in QAOA, often chosen as pi
INITIAL_BETA = np.pi/2 # Initial value for the beta parameter in QAOA, often chosen as pi/2

**Create Cost Modularity Hamiltonian**

$H_{\text{mod}}=-\frac{1}{4m}\sum_{r=1}^{k}\sum_{i<j} B_{ij}\left(1 - Z_{i,r} - Z_{j,r} + Z_{i,r}Z_{j,r}\right)$



In [ ]:
def create_cost_modularity_hamiltonian(G, B, N, K):
    """Creates the Modularity Penaty (H_mod) for the graph modularity problem.

    The Hamiltonian penalizes assignments that do not align with the
    modularity metric. It is derived from the modularity matrix B.

    Args:
        G (networkx.Graph): The original graph for which the modularity
                            matrix and Cost Hamiltonian are derived.
        B (numpy.ndarray): The modularity matrix B for the graph G.
        N (int): The number of nodes in the original graph.
        K (int): The number of communities.
        
    Returns:
        tuple: A tuple containing:
            - num_qubits (int): The total number of qubits required for the problem (n*k).
            - cost_hamiltonian (qiskit.quantum_info.SparsePauliOp): The constructed
                                                                    Cost Hamiltonian.
    """
    modularity_penalty_pauli_terms = []

    # m: Total number of edges in the graph G
    m = G.number_of_edges()

    # Total number of qubits: n (nodes in G) * k (number of communities)
    # This mapping means each original node 'i' and community 'r' corresponds to a unique qubit (i*k + r).
    num_qubits = N * K

    # Ensure m is not zero to avoid division by zero in coefficients
    if m == 0:
        print("Graph has no edges. The Modularity Hamiltonian will be trivial (zero).")
        # In this case, the Hamiltonian is effectively 0.
        modularity_penalty_pauli_terms = SparsePauliOp.from_list([], num_qubits=num_qubits)
    else:
        # Calculate sum_j B_ij for the linear terms
        row_sums_B = np.sum(B, axis=1)

        # Calculate sum_{i<j} B_ij for the constant term
        sum_B_ij_i_lt_j = 0.0
        for i_idx in range(N):
            for j_idx in range(i_idx + 1, N):
                sum_B_ij_i_lt_j += B[i_idx, j_idx]

        # 1. Constant Term: - (k / (4m)) * sum_{i<j} B_ij
        # This term is added to ensure the Hamiltonian is shifted correctly.
        coeff_const = (-K / (4 * m)) * sum_B_ij_i_lt_j
        if coeff_const != 0:
            modularity_penalty_pauli_terms.append(('I' * num_qubits, coeff_const))

        for r in range(K): # Iterate over communities (0 to K-1)
            for i in range(N): # Iterate over original nodes i (0 to N-1)
                # 2. Linear Terms: (1 / (4m)) * (sum_{j!=i} B_ij) * Z_ir
                # This term accounts for connections of node 'i' to all other nodes in the graph.
                # sum_{j!=i} B_ij = (sum_j B_ij) - B_ii = row_sums_B[i] - B[i,i]
                sum_B_xj_j_neq_x = row_sums_B[i] - B[i, i]
                coeff_linear = sum_B_xj_j_neq_x / (4 * m)

                if coeff_linear != 0:
                    qubit_idx_ir = i * K + r # Map node i, community r to a unique qubit index
                    pauli_string_linear = ['I'] * num_qubits
                    pauli_string_linear[qubit_idx_ir] = 'Z' # Apply Z operator on the corresponding qubit
                    modularity_penalty_pauli_terms.append(("".join(reversed(pauli_string_linear)), coeff_linear))

                for j in range(i + 1, N): # Iterate over original nodes j (0 to N-1), ensuring i < j
                    # 3. Quadratic Terms: - (B_ij / (4m)) * Z_ir Z_jr
                    # This term accounts for interactions between node 'i' and node 'j' if they are
                    # in the same community 'r'.
                    coeff_quadratic = -B[i, j] / (4 * m)

                    if coeff_quadratic != 0:
                        qubit_idx_ir = i * K + r # Qubit for node i in community r
                        qubit_idx_jr = j * K + r # Qubit for node j in community r
                        pauli_string_quadratic = ['I'] * num_qubits
                        pauli_string_quadratic[qubit_idx_ir] = 'Z'
                        pauli_string_quadratic[qubit_idx_jr] = 'Z' # Apply Z operators on corresponding qubits
                        modularity_penalty_pauli_terms.append(("".join(reversed(pauli_string_quadratic)), coeff_quadratic))

        # Create the SparsePauliOp from the list of terms. Pass num_qubits to handle edge cases.
        modularity_penalty_hamiltonian = SparsePauliOp.from_list(modularity_penalty_pauli_terms, num_qubits=num_qubits)

    print(f"\n\nModularity Penalty Hamiltonian (Hz) constructed for {num_qubits} qubits:")
    print(modularity_penalty_hamiltonian)
    print(f"Number of terms in Hamiltonian: {len(modularity_penalty_pauli_terms)}")

    return num_qubits, modularity_penalty_hamiltonian

**Create One-Hot Penalty Hamiltonian**

$
H_{\text{pen}} = \sum_{i} \left\{ \frac{\lambda}{2} \sum_{r<s} Z_{i,r} Z_{i,s} \right. - \left. \frac{\lambda (k-2)}{2} \sum_{r=1}^{k} Z_{i,r} \right\}
$


In [ ]:
def create_onehot_penalty_hamiltonian(G, num_qubits, K, LAMBDA_PENALTY):
    """Creates the One-Hot Penalty Hamiltonian, which enforces the one-hot encoding constraint.

    The one-hot encoding means that for each node, exactly one qubit corresponding
    to its community assignment should be in the |1> state, and all others in |0>.
    This Hamiltonian penalizes configurations where a node is assigned to zero
    or multiple communities.

    Args:
        G (networkx.Graph): The original graph (used for `n` nodes).
        num_qubits (int): Total number of qubits in the system (n*k).
        K (int): The number of communities, which determines the structure of the penalty terms.
        LAMBDA_PENALTY (float): The penalty coefficient that determines the strength of the penalty terms.

    Returns:
        qiskit.quantum_info.SparsePauliOp: The constructed Penalty Hamiltonian.
    """
    onehot_penalty_pauli_terms = []

    if K < 2:
        print("One-Hot Penalty Hamiltonian requires at least two communities (k >= 2).")
        onehot_penalty_pauli_terms = SparsePauliOp.from_list([], num_qubits=num_qubits)
    else:
        for i in range(N): # Iterate over each original node (0 to N-1)
            # Term 1: (lambda/2) * sum_{r<s} Z_ir Z_is
            # This term penalizes a node 'i' being assigned to two different communities 'r' and 's' simultaneously.
            for r in range(K): # Iterate over community r
                for s in range(r + 1, K): # Iterate over community s, where s > r
                    qubit_idx_ir = i * K + r # Qubit for node i in community r
                    qubit_idx_is = i * K + s # Qubit for node i in community s

                    pauli_string_quadratic = ['I'] * num_qubits
                    pauli_string_quadratic[qubit_idx_ir] = 'Z'
                    pauli_string_quadratic[qubit_idx_is] = 'Z'

                    coeff_quadratic = LAMBDA_PENALTY / 2.0
                    onehot_penalty_pauli_terms.append(("".join(reversed(pauli_string_quadratic)), coeff_quadratic))

            # Term 2: - (lambda * (k-2) / 2) * sum_{r=1 to k} Z_ir
            # This term ensures that for each node, the sum of Z_ir operators biases towards a single assignment.
            # In conjunction with Term 1, this results in the one-hot constraint (exactly one qubit is |1>).
            for r in range(K): # Iterate over community r
                qubit_idx_ir = i * K + r # Qubit for node i in community r

                pauli_string_linear = ['I'] * num_qubits
                pauli_string_linear[qubit_idx_ir] = 'Z'

                coeff_linear = -LAMBDA_PENALTY * (K - 2) / 2.0
                onehot_penalty_pauli_terms.append(("".join(reversed(pauli_string_linear)), coeff_linear))

        # Create the SparsePauliOp from the list of terms.
        onehot_penalty_hamiltonian = SparsePauliOp.from_list(onehot_penalty_pauli_terms, num_qubits=num_qubits)

    print(f"\n\nOne Hot Penalty Hamiltonian constructed for {num_qubits} qubits:")
    print(onehot_penalty_hamiltonian)
    print(f"Number of terms in Penalty Hamiltonian: {len(onehot_penalty_pauli_terms)}")
    return onehot_penalty_hamiltonian

In [ ]:
def create_total_cost_hamiltonian(cost_hamiltonian, penalty_hamiltonian, num_qubits):
      """Combines the Cost Hamiltonian and the Penalty Hamiltonian to form the total objective function.

      The total Hamiltonian represents the overall optimization problem, where the goal is to
      find a quantum state that minimizes its expectation value. This combined Hamiltonian
      includes terms that encourage desired community structures (Cost Hamiltonian) and
      terms that enforce problem constraints (Penalty Hamiltonian).

      Args:
            cost_hamiltonian (qiskit.quantum_info.SparsePauliOp): The Hamiltonian representing the modularity objective.
            penalty_hamiltonian (qiskit.quantum_info.SparsePauliOp): The Hamiltonian enforcing the one-hot encoding constraint.
            
      Returns:
            qiskit.quantum_info.SparsePauliOp: The combined total Hamiltonian.
      """
      # Combine the Cost Hamiltonian and Penalty Hamiltonian by summing them.
      # The objective is to minimize this combined Hamiltonian.
      total_hamiltonian = cost_hamiltonian + penalty_hamiltonian

      print(f"Total Hamiltonian constructed for {num_qubits} qubits:")
      print(total_hamiltonian)
      print(f"Number of terms in Total Hamiltonian: {len(total_hamiltonian)}")

      return total_hamiltonian

In [ ]:
def run_one_hot(G, N, K, P, LAMBDA_PENALTY):
    """Runs the full QAOA pipeline for community detection using one-hot encoding.

    Constructs the supernode graph, modularity cost Hamiltonian, one-hot penalty
    Hamiltonian, XY mixer, and the QAOA ansatz. Runs classical optimisation on a
    local AerSimulator, then samples the optimised circuit and decodes the result
    into community assignments.

    Args:
        G (networkx.Graph): The problem graph.
        N (int): The number of nodes in the graph.
        K (int): The number of communities.
        P (int): The number of QAOA layers.
        LAMBDA_PENALTY (float): Penalty coefficient for the one-hot constraint Hamiltonian.
    """
    if K > N:
        K = N # Clamp K to the number of nodes to keep the problem well-defined

    display_graph(G, "Problem Graph") # Visualise the initial problem graph

    G_onehot_supernodes_labeled = create_supernode(G, K) # Create the one-hot encoded supernode graph
    display_graph(G_onehot_supernodes_labeled, "Supernode Structure") # Visualise the supernode structure

    modularity_matrix = create_modularity_matrix(G, N) # Calculate the modularity matrix for the graph

    # Check if the graph can be partitioned into communities using spectral methods before proceeding with QAOA
    is_community_detectable = not spectral_stop(modularity_matrix)
    if not is_community_detectable:
        print("The graph cannot be partitioned into communities based on spectral methods. Exiting.")
        return

    num_qubits, cost_hamiltonian = create_cost_modularity_hamiltonian(G, modularity_matrix, N, K) # Construct the Cost Hamiltonian (modularity-based)

    penalty_hamiltonian = create_onehot_penalty_hamiltonian(G, num_qubits, K, LAMBDA_PENALTY) # Construct the Penalty Hamiltonian (one-hot constraint-based)

    total_hamiltonian = create_total_cost_hamiltonian(cost_hamiltonian, penalty_hamiltonian, num_qubits) # Combine cost and penalty to form the total objective Hamiltonian

    mixer_hamiltonian_xy = create_xy_mixer_hamiltonian(num_qubits, N, K) # Construct the XY Mixer Hamiltonian for QAOA

    # QAOA ansatz construction
    circuit = create_qaoa_ansatz(total_hamiltonian, P, N, K, mixer_hamiltonian=mixer_hamiltonian_xy, initial_state="W") # Create the QAOA ansatz with W-state initialisation and XY mixer
    print(f"\n\nQAOA Ansatz: ")
    draw_circuit_mpl(circuit, "QAOA Ansatz Circuit") # Draw and display the QAOA ansatz circuit

    # Optimisation process
    initial_params = [INITIAL_GAMMA, INITIAL_BETA] * P # Set initial parameters for the QAOA optimisation
    transpiled_qc, backend = run_ansatz_simulator_circuit_prep(circuit, method="statevector") # Prepare the circuit for statevector simulation (transpile for AerSimulator)
    result, value_tracker = run_optimizer_simulator(transpiled_qc, backend, initial_params, total_hamiltonian) # Run the classical optimiser to find optimal QAOA parameters
    plot_optimizer_results(value_tracker) # Plot the cost function's value over optimisation iterations

    # Result extraction and interpretation
    optimized_circuit = transpiled_qc.assign_parameters(result.x) # Assign the optimised parameters to the quantum circuit
    optimized_circuit.measure_all() # Add measurements to all qubits for SamplerV2

    print(f"\n\nThe circuit with optimized parameters: ")
    draw_circuit_mpl(optimized_circuit, "Optimized Circuit") # Draw and display the optimised circuit

    # Sample measurement outcomes from the optimised circuit to get the solution bitstring
    optimized_output = sample_output(optimized_circuit, backend, num_qubits)

    print(f"\n\nThe optimized bitstring for the problem in hand: ")
    final_answer = [str(i) for i in optimized_output]
    print(final_answer)

    print(f"\n\nThe community division w.r.t. the input k = {K}:")
    node_colors = group_nodes_by_community(G, final_answer, K, N) # Group nodes into communities based on the final bitstring and visualise the result
    display_graph(G, "Graph Partitioned into Communities", node_colors, 42) # Display the final graph with colour-coded community assignments

In [ ]:

K = 2  # Number of communities
P = 1  # Number of QAOA layers
LAMBDA_PENALTY = 1# Penalty parameter for the Hamiltonian to enforce constraints

G, N = create_clique_chain_graph(3, 8)

run_one_hot(G, N, K, P, LAMBDA_PENALTY)

**Classical Computation**

$H_C = \max_{\mathbf{x}} \left( \sum_{j=1}^{k} \mathbf{x}_j^T B \mathbf{x}_j + \sum_{i=1}^{n} \gamma_i \left( \sum_{j=1}^{k} x_{i,j} - 1 \right)^2 \right)$

In [ ]:
'''
Objective (maximisation):
  Σ_j  x_j^T B x_j  +  Σ_i γ_i (Σ_j x_{i,j} - 1)²
γ_i = -LAMBDA_PENALTY  (negative so one-hot violations reduce the objective)

Variables: X[i, j] = x_{i,j}  (continuous relaxation of binary assignment)
Strategy : continuous relaxation → COBYLA → argmax decode → visualise
'''

from scipy.optimize import minimize


def classical_objective(x_flat, B, N, K, lambda_penalty, tracker):
    """Computes the classical objective for the continuous relaxation of the modularity problem.

    Evaluates the modularity gain minus a penalty for violating the one-hot
    constraint. The value is negated so that minimisation routines can be
    used to maximise the true objective.

    Args:
        x_flat (numpy.ndarray): Flattened assignment matrix of shape (n*k,).
        B (numpy.ndarray): The modularity matrix for the graph.
        n (int): The number of nodes in the graph.
        k (int): The number of communities.
        lam (float): Penalty coefficient for one-hot constraint violations.
        tracker (list): Accumulates the objective value at each evaluation for convergence plots.

    Returns:
        float: Negated objective value (for use with minimisation routines).
    """
    X = x_flat.reshape(N, K) # Reshape flat array into (N x K) assignment matrix

    modularity = sum(X[:, j] @ B @ X[:, j] for j in range(K)) # Sum of modularity contributions across all communities
    penalty = -lambda_penalty * np.sum((X.sum(axis=1) - 1) ** 2) # Penalise rows that do not sum to 1 (γ_i = -lambda_penalty)
    value = modularity + penalty

    tracker.append(value) # Record the current objective value for convergence tracking
    return -value # Negate to convert maximisation to minimisation


def run_classical_optimizer(B, N, K, lambda_penalty, n_restarts=10, seed=42):
    """Optimises the classical objective using COBYLA with multiple random restarts.

    Uses a continuous relaxation of the binary community assignment problem.
    Multiple Dirichlet-initialised restarts reduce the risk of converging to
    a poor local optimum. The best result across all restarts is returned.

    Args:
        B (numpy.ndarray): The modularity matrix for the graph.
        N (int): The number of nodes in the graph.
        K (int): The number of communities.
        LAMBDA_PENALTY (float): Penalty coefficient for the one-hot constraint.
        n_restarts (int): Number of independent random restarts.
        seed (int): Random seed for reproducibility.

    Returns:
        tuple: A tuple containing:
            - best_result (scipy.optimize.OptimizeResult): The best optimisation result found.
            - tracker (list[float]): Objective values recorded across all evaluations.
    """
    rng = np.random.default_rng(seed)
    best_result = None
    tracker = []

    # non-negativity constraints: x_{i,j} >= 0  for all i, j
    constraints = [
        {"type": "ineq", "fun": lambda x, idx=idx: x[idx]}
        for idx in range(N * K)
    ]

    for _ in range(n_restarts):
        # Dirichlet initialisation: rows sum to 1, all entries >= 0
        x0 = rng.dirichlet(np.ones(K), size=N).flatten()

        result = minimize(
            classical_objective,
            x0,
            args=(B, N, K, lambda_penalty, tracker),
            method="COBYLA",
            constraints=constraints,
            tol=1e-6,
            options={"maxiter": 5000, "rhobeg": 0.5},
        )

        if best_result is None or result.fun < best_result.fun:
            best_result = result # Keep the result with the lowest (negated) objective value

    return best_result, tracker


def decode_assignments(x_flat, N, K):
    """Decodes the continuous relaxation solution into integer community assignments.

    For each node, selects the community with the highest assignment value,
    converting the continuous solution into a hard (discrete) partitioning.

    Args:
        x_flat (numpy.ndarray): Flattened assignment matrix of shape (n*k,).
        n (int): The number of nodes in the graph.
        k (int): The number of communities.

    Returns:
        numpy.ndarray: Integer array of shape (n,) giving the community index for each node.
    """
    return np.argmax(x_flat.reshape(N, K), axis=1)  # Community with the highest assignment value for each node


def plot_classical_convergence(tracker):
    """Plots the objective value over function evaluations during COBYLA optimisation.

    Args:
        tracker (list[float]): Sequence of objective values recorded during optimisation.
    """
    plt.figure(figsize=(10, 4))
    plt.plot(tracker)
    plt.xlabel("Function evaluation")
    plt.ylabel("Objective value")
    plt.title("Classical One-Hot — objective per evaluation")
    plt.tight_layout()
    plt.show()


def plot_classical_communities(G, assignments):
    """Visualises the detected community structure of the graph using colour-coded nodes.

    Args:
        G (networkx.Graph): The original problem graph.
        assignments (numpy.ndarray): Integer array of community indices for each node.
    """
    palette = [
        '#e6194b', # Community 0 color
        '#3cb44b', # Community 1 color
        '#ffe119', # Community 2 color
        '#4363d8',
        '#f58231',
        '#911eb4',
        '#46f0f0',
        '#f032e6',
        '#bcf60c',
        '#fabebe',
        '#008080',
        '#e6beff',
        '#9a6324',
        '#fffac8',
        '#800000',
        '#aaffc3',
        '#808000',
        '#ffd8b1',
        '#000075',
        '#808080',
        '#ffffff',
        '#000000'
    ]
    node_colors = [palette[c % len(palette)] for c in assignments] # Assign a colour to each node based on its community
    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos=pos, node_color=node_colors, with_labels=True,
            node_size=700, font_color="white")
    plt.title("Detected Communities")
    plt.show()


def onehot_classical_optimization(G, N, K, lambda_penalty):
    """Runs the full classical COBYLA optimisation pipeline for the one-hot community detection problem.

    Constructs the graph and modularity matrix, runs the continuous-relaxation
    COBYLA optimiser, decodes the result into community assignments, and
    displays the convergence plot and the final community partition.
    """
    display_graph(G, "Input Graph") # Visualise the input graph
    B = create_modularity_matrix(G, N) # Calculate the modularity matrix for the graph

    result, tracker = run_classical_optimizer(B, N, K, lambda_penalty) # Run COBYLA optimisation with multiple random restarts
    assignments      = decode_assignments(result.x, N, K) # Decode continuous solution into hard community assignments

    print(f"Optimal objective : {-result.fun:.4f}")
    print(f"Community assignments: {assignments}")
    print(f"Function evaluations : {result.nfev}")

    plot_classical_convergence(tracker) # Plot the objective value over optimisation evaluations
    plot_classical_communities(G, assignments) # Visualise the final community partition on the graph

G, N = get_davis_southern_women_graph() 
K = 2  # Number of communities
LAMBDA_PENALTY = 30# Penalty parameter for the Hamiltonian to enforce constraints

print(LAMBDA_PENALTY, K, N)

onehot_classical_optimization(G, N, K, LAMBDA_PENALTY)